# Vignette de plaques réemployées : Salomon / Solis

Cas d'étude ponctuel, distinct de `03_reemploi_plaques.ipynb` (qui couvre tout le corpus) :
un même jeu de plaques gravées — celui de **Bernard Salomon** (Lyon, 1557), repris **à
l'identique mais inversé (image en miroir)** par **Virgil Solis** (s.l., 1563) — est réemployé
dans de nombreuses éditions ultérieures. L'idée : plutôt qu'une frise ou une carte, montrer la
vignette elle-même (segmentée automatiquement, cf. `notebooks/gallica_utils.py`) au centre,
entourée des pages qui la réemploient.

**Colonne source** : ma collègue a ajouté `à utiliser pour visu 3` dans `BNU_corpus.ods`
(feuille `Synthèse`) — une page Gallica/MDZ/HathiTrust/Manuscriptorium par édition concernée.
Les deux pages ci-dessous sont les pages sources, celles à segmenter pour obtenir les deux
vignettes centrales (mêmes plaques, orientation inversée) :

- **Salomon** : https://gallica.bnf.fr/ark:/12148/btv1b2200047r/f10.item
- **Solis** : https://gallica.bnf.fr/ark:/12148/btv1b8454722p/f28.item

**Interaction** : la vignette centrale peut être Salomon ou Solis. Autour d'elle, deux arcs de
pages (celles qui utilisent les plaques Salomon à gauche, celles qui utilisent les plaques
Solis à droite) — chacune affichée en entier (pas recadrée). Cliquer une page recentre la
vignette correspondant à sa famille de plaques.

In [1]:
import os
import re
import json
import base64
from io import BytesIO
import urllib.request
import urllib.error

from odf.opendocument import load as charger_ods
from odf.table import Table, TableRow, TableCell
from odf.text import P as OdfP
from PIL import Image

## 1. Chargement des pages concernées

Même lecteur ODS cellule par cellule que dans les autres notebooks (le lecteur ODF de
`pandas.read_excel` ignore silencieusement certaines colonnes). On ne garde que les lignes où
`à utiliser pour visu 3` est renseignée, **en écartant les liens Dropbox** (deux cas : la page
n'a pas encore de version en ligne consultable directement, ma collègue a donné un PDF à la
place — non exploitable ici de la même façon).

In [2]:
def lire_feuille_ods(chemin, nom_feuille):
    """Lit une feuille ODS cellule par cellule (contourne les colonnes ignorées par pandas)."""
    doc = charger_ods(chemin)
    feuille = next(t for t in doc.spreadsheet.getElementsByType(Table)
                   if t.getAttribute("name") == nom_feuille)
    lignes = feuille.getElementsByType(TableRow)

    def texte_cellule(cellule):
        return "\n".join(
            "".join(n.data for n in p.childNodes if n.nodeType == 3)
            for p in cellule.getElementsByType(OdfP)
        )

    def cellules_ligne(ligne):
        valeurs = []
        for cellule in ligne.getElementsByType(TableCell):
            repet = int(cellule.getAttribute("numbercolumnsrepeated") or 1)
            valeurs.extend([texte_cellule(cellule)] * repet)
        return valeurs

    entetes = cellules_ligne(lignes[0])
    return entetes, [cellules_ligne(l) for l in lignes[1:]]


entetes, lignes = lire_feuille_ods("../../retours_celine/BNU_corpus.ods", "Synthèse")
idx = {
    "ville": entetes.index("ville"),
    "annee": entetes.index("année"),
    "titre": entetes.index("titre abrégé"),
    "publisher": entetes.index("publisher"),
    "graveur": entetes.index("graveur\xa0: Nom, Prénom"),
    "visu3": entetes.index("à utiliser pour visu 3"),
}

pages_brutes = []
for l in lignes:
    if idx["visu3"] >= len(l):
        continue
    valeur = l[idx["visu3"]].strip()
    if not valeur or "dropbox.com" in valeur.lower():
        continue
    url = next((m.strip() for m in valeur.split() if m.strip().startswith("http")), None)
    if not url:
        continue
    pages_brutes.append({
        "ville": l[idx["ville"]].strip(),
        "annee": int(l[idx["annee"]].strip()),
        "titre": l[idx["titre"]].strip(),
        "publisher": l[idx["publisher"]].strip() or "Éditeur non identifié",
        "graveur": l[idx["graveur"]].strip(),
        "url": url,
    })

print(f"{len(pages_brutes)} pages retenues (hors liens Dropbox)")
for p in pages_brutes:
    print(f"  {p['annee']} . {p['ville']:25s} . {p['graveur']:16s} . {p['url']}")

10 pages retenues (hors liens Dropbox)
  1557 . Lyon                      . Salomon, Bernard . https://gallica.bnf.fr/ark:/12148/btv1b2200047r/f10.item
  1559 . Lyon                      . Salomon, Bernard . https://gallica.bnf.fr/ark:/12148/btv1b22000485/f11.item
  1563 . s.l.                      . Solis, Virgil    . https://gallica.bnf.fr/ark:/12148/btv1b8454722p/f28.item
  1564 . Francfort-sur-le-Main     . Solis, Virgil    . https://www.digitale-sammlungen.de/en/view/bsb00034313?page=51
  1569 . Francfort                 . Solis, Virgil    . https://gallica.bnf.fr/ark:/12148/btv1b2200054w/f8.item
  1570 . Paris                     . Salomon, Bernard . https://gallica.bnf.fr/ark:/12148/bpt6k8708104s/f35.item
  1571 . Francfort                 . Solis, Virgil    . https://babel.hathitrust.org/cgi/pt?id=emu.010002482229&seq=33&view=1up
  1574 . Paris                     . Salomon, Bernard . https://gallica.bnf.fr/ark:/12148/bpt6k8710284z/f38.item
  1595 . Anvers                    . 

## 2. Identification de la famille de plaques (Salomon / Solis)

La colonne `graveur` de chaque ligne suffit à classer chaque page dans l'une des deux
familles — sans ambiguïté : les 10 pages retenues se répartissent en 4 pages « Salomon » et
6 pages « Solis » (dont, dans chaque famille, la page source elle-même).

In [3]:
def famille_depuis_graveur(graveur):
    g = graveur.lower()
    if "salomon" in g:
        return "salomon"
    if "solis" in g:
        return "solis"
    return None


ARK_SOURCE_SALOMON = "btv1b2200047r"
ARK_SOURCE_SOLIS = "btv1b8454722p"

for i, p in enumerate(pages_brutes):
    p["id"] = f"p{i}"
    p["famille"] = famille_depuis_graveur(p["graveur"])
    p["source"] = (ARK_SOURCE_SALOMON in p["url"]) or (ARK_SOURCE_SOLIS in p["url"])

assert all(p["famille"] for p in pages_brutes), "Page sans famille Salomon/Solis identifiable"

par_famille = {"salomon": [p for p in pages_brutes if p["famille"] == "salomon"],
               "solis": [p for p in pages_brutes if p["famille"] == "solis"]}
print(f"Salomon : {len(par_famille['salomon'])} pages | Solis : {len(par_famille['solis'])} pages")

Salomon : 4 pages | Solis : 6 pages


## 3. Téléchargement des pages

Aucun code existant dans le dépôt ne récupérait jusqu'ici une page Gallica ou MDZ en image
(seule la métadonnée Gallica est utilisée ailleurs, via l'API de recherche BnF). Deux motifs
d'URL, vérifiés directement :

- **Gallica** — une URL de visionneuse `.../ark:/12148/{ark}/f{n}.item` donne, via l'API IIIF
  Image de Gallica, `https://gallica.bnf.fr/iiif/ark:/12148/{ark}/f{n}/full/full/0/native.jpg`.
- **digitale-sammlungen.de (MDZ)** — une URL de visionneuse `.../view/{id}?page={n}` donne,
  via l'API IIIF Image de la MDZ, `https://api.digitale-sammlungen.de/iiif/image/v2/{id}_{n:05d}/full/full/0/default.jpg`.

Pour **HathiTrust** et **Manuscriptorium**, aucun motif équivalent n'a été trouvé : HathiTrust
bloque les requêtes hors navigateur (protection Cloudflare, `403` systématique même avec un
en-tête `User-Agent` de navigateur) et aucun code du dépôt ne traite Manuscriptorium. Ces deux
pages restent donc **liens externes uniquement** dans la visualisation (pas de vignette, un
lien "voir la page" vers le site d'origine) — traitement identique à celui déjà appliqué
ailleurs dans le projet pour les pages Gallica non numérisées (403).

In [4]:
def image_gallica(url):
    m = re.search(r"ark:/12148/(\w+)/f(\d+)", url)
    if not m:
        return None
    ark, folio = m.group(1), m.group(2)
    return f"https://gallica.bnf.fr/iiif/ark:/12148/{ark}/f{folio}/full/full/0/native.jpg"


def image_mdz(url):
    m_id = re.search(r"/view/(\w+)", url)
    m_page = re.search(r"page=(\d+)", url)
    if not (m_id and m_page):
        return None
    return f"https://api.digitale-sammlungen.de/iiif/image/v2/{m_id.group(1)}_{int(m_page.group(1)):05d}/full/full/0/default.jpg"


for p in pages_brutes:
    if "gallica.bnf.fr" in p["url"]:
        p["image_url"] = image_gallica(p["url"])
    elif "digitale-sammlungen.de" in p["url"]:
        p["image_url"] = image_mdz(p["url"])
    else:
        p["image_url"] = None  # HathiTrust, Manuscriptorium : lien externe uniquement

sans_image = [p for p in pages_brutes if p["image_url"] is None]
print(f"{len(pages_brutes) - len(sans_image)} pages avec image récupérable, {len(sans_image)} en lien externe uniquement :")
for p in sans_image:
    print(f"  {p['annee']} . {p['titre']} ({p['url']})")

8 pages avec image récupérable, 2 en lien externe uniquement :
  1571 . Pub. Ouidii Nasonis Metamorphoseon libri XV (https://babel.hathitrust.org/cgi/pt?id=emu.010002482229&seq=33&view=1up)
  1595 . Las Transformaciones de Ovidio (https://www.manuscriptorium.com/hub/browser/default/detail?url=https:%2F%2Fcollectiones.manuscriptorium.com%2Fassorted%2FUCM___%2FUCM___%2F8%2FUCM___-UCM___BH_FLL_308332NO62V8-es%2F&lang=cs&imageId=https:%2F%2Fimagines.manuscriptorium.com%2Floris%2FUCM___-UCM___BH_FLL_308332NO62V8-es%2Fid_719465_7r)


In [5]:
import time

CACHE_DIR = "cache_visu5_pages"
os.makedirs(CACHE_DIR, exist_ok=True)

def telecharger_avec_reprises(url, chemin, tentatives=5):
    for i in range(tentatives):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=30) as reponse, open(chemin, "wb") as f:
                f.write(reponse.read())
            return
        except urllib.error.HTTPError as e:
            if e.code == 429 and i < tentatives - 1:
                attente = 5 * (i + 1)
                print(f"  429 (trop de requetes), nouvelle tentative dans {attente}s...")
                time.sleep(attente)
            else:
                raise

for p in pages_brutes:
    if p["image_url"] is None:
        p["chemin_local"] = None
        continue
    chemin = os.path.join(CACHE_DIR, p["id"] + ".jpg")
    if not os.path.exists(chemin):
        telecharger_avec_reprises(p["image_url"], chemin)
        print(f"telecharge : {chemin}")
        time.sleep(1.5)
    p["chemin_local"] = chemin

print("OK -", sum(p["chemin_local"] is not None for p in pages_brutes), "images en cache")

telecharge : cache_visu5_pages/p7.jpg


telecharge : cache_visu5_pages/p9.jpg


OK - 8 images en cache


## 4. Segmentation des deux vignettes centrales

Même modèle YOLO que dans `notebooks/gallica_utils.py` (déjà utilisé par
`classification_bois_cuivre` et `classification_graveur`), appliqué ici seulement aux deux
pages sources.

- **Salomon** (Lyon, 1557) — détection automatique nette : une seule illustration détectée,
  confiance 0.86, cadrée exactement sur la scène (sans le texte ni le cadre orné).
- **Solis** (s.l., 1563) — la détection automatique échoue à isoler l'illustration : au seuil
  par défaut (0.25) aucune détection ; en abaissant le seuil, une seule détection apparaît mais
  à confiance très faible (0.14) et englobe tout le cadre ornemental (angelots, urnes,
  rinceaux) autour de la scène, contrairement au recadrage net obtenu pour Salomon. Après
  vérification qu'aucune détection plus resserrée n'existe à aucun seuil (les boîtes
  candidates avant NMS ont été inspectées une à une), la scène a été **recadrée manuellement**
  par inspection visuelle directe de la page — cas ponctuel, justifié ici par le fait qu'il ne
  s'agit que de 2 pages précises (pas d'un traitement à généraliser à tout le corpus, où
  l'automatique reste la règle, cf. `notebooks/gallica_utils.py`).

In [6]:
import sys

RACINE = os.path.abspath("../..")
sys.path.insert(0, os.path.join(RACINE, "notebooks"))
from gallica_utils import charger_yolo, segmenter_page, liberer_yolo

page_salomon = next(p for p in pages_brutes if ARK_SOURCE_SALOMON in p["url"])
page_solis = next(p for p in pages_brutes if ARK_SOURCE_SOLIS in p["url"])

DOSSIER_VIGNETTES = "cache_visu5_vignettes"
os.makedirs(DOSSIER_VIGNETTES, exist_ok=True)

modele_yolo = charger_yolo(yolov5_repo=os.path.join(RACINE, "yolov5_repo"))
nb_detections = segmenter_page(page_salomon["chemin_local"], "salomon", DOSSIER_VIGNETTES, modele_yolo, conf_thres=0.25)
print(f"Salomon : {nb_detections} illustration(s) detectee(s)")
modele_yolo = liberer_yolo(modele_yolo)

✓ YOLO chargé — classes : {0: 'illustration'}


Salomon : 1 illustration(s) detectee(s)
✓ Mémoire GPU libérée


In [7]:
fichier_vignette_salomon = next(
    f for f in os.listdir(DOSSIER_VIGNETTES) if f.startswith("salomon_det")
)
vignette_salomon = Image.open(os.path.join(DOSSIER_VIGNETTES, fichier_vignette_salomon)).convert("RGB")
print("vignette Salomon :", fichier_vignette_salomon, vignette_salomon.size)

# Solis : recadrage manuel (cf. section 4 ci-dessus), coordonnees etablies par inspection
# visuelle directe de la page source pleine resolution (2966x2542).
img_solis = Image.open(page_solis["chemin_local"]).convert("RGB")
assert img_solis.size == (2966, 2542), (
    f"Taille inattendue pour la page Solis : {img_solis.size} "
    "(le recadrage manuel ci-dessous suppose cette resolution ; si Gallica change sa sortie "
    "IIIF par defaut, ces coordonnees ne seront plus valables)"
)
BOITE_SOLIS_MANUELLE = (745, 745, 2175, 1610)
vignette_solis = img_solis.crop(BOITE_SOLIS_MANUELLE)
vignette_solis.save(os.path.join(DOSSIER_VIGNETTES, "solis_recadre_manuel.jpg"), quality=92)
print("vignette Solis (recadree manuellement) :", vignette_solis.size)

vignette Salomon : salomon_det1_conf0.86.jpg (592, 455)
vignette Solis (recadree manuellement) : (1430, 865)


## 5. Mise en page (HTML autonome)

Disposition en deux arcs autour du centre : les pages « famille Salomon » à gauche, les pages
« famille Solis » à droite — l'opposition gauche/droite fait écho au miroir entre les deux
jeux de plaques. Chaque page est affichée **en entier** (pas recadrée), sauf les deux pages
sans image récupérable (HathiTrust, Manuscriptorium) montrées comme une carte texte avec lien
externe. Cliquer une page recentre la vignette de sa famille ; comme dans les autres
notebooks, le survol seul montre un aperçu sans lien (le lien "voir" n'apparaît qu'une fois la
carte cliquée, cf. `contenuApercu`/`contenuDetaille` dans `02_nuage_editions_villes.ipynb`).

Toutes les images (vignettes + pages) sont ré-encodées en base64 et intégrées directement dans
le HTML : un fichier unique et autonome, comme les 4 autres notebooks, malgré la présence
d'images (contrairement à `01`-`04`, purement vectoriels).

In [8]:
def encoder_base64(img, largeur_max, qualite):
    img = img.convert("RGB")
    if img.width > largeur_max:
        ratio = largeur_max / img.width
        img = img.resize((largeur_max, round(img.height * ratio)), Image.LANCZOS)
    tampon = BytesIO()
    img.save(tampon, format="JPEG", quality=qualite)
    return "data:image/jpeg;base64," + base64.b64encode(tampon.getvalue()).decode()


vignette_salomon_b64 = encoder_base64(vignette_salomon, largeur_max=900, qualite=88)
vignette_solis_b64 = encoder_base64(vignette_solis, largeur_max=900, qualite=88)

for p in pages_brutes:
    if p["chemin_local"]:
        img = Image.open(p["chemin_local"])
        p["img"] = encoder_base64(img, largeur_max=380, qualite=74)
    else:
        p["img"] = None

taille_totale_ko = sum(len(p["img"] or "") for p in pages_brutes) // 1024
print(f"images des pages encodees : ~{taille_totale_ko} Ko")

images des pages encodees : ~625 Ko


In [9]:
import math

LARGEUR_SCENE, HAUTEUR_SCENE = 1100, 900
CX, CY = LARGEUR_SCENE / 2, HAUTEUR_SCENE / 2 - 10
RAYON = 380


def placer_arc(items, angle_debut, angle_fin):
    n = len(items)
    for i, p in enumerate(items):
        t = 0.5 if n == 1 else i / (n - 1)
        angle = math.radians(angle_debut + t * (angle_fin - angle_debut))
        p["x"] = round(CX + RAYON * math.cos(angle), 1)
        p["y"] = round(CY + RAYON * math.sin(angle), 1)


placer_arc(sorted(par_famille["salomon"], key=lambda p: p["annee"]), 100, 260)
placer_arc(sorted(par_famille["solis"], key=lambda p: p["annee"]), -75, 75)

for p in pages_brutes:
    p["lien_html_ok"] = p["url"].startswith("http")

print("positions calculees pour", len(pages_brutes), "pages")

positions calculees pour 10 pages


In [10]:
def carte_page_html(p):
    if p["img"]:
        contenu = f'<img src="{p["img"]}" alt="{p["titre"]}">'
    else:
        contenu = '<div class="pas-image">image non disponible<br>(acces restreint)</div>'
    marque_source = ' data-source="1"' if p["source"] else ""
    return (
        f'<div class="carte-page famille-{p["famille"]}" id="{p["id"]}" data-i="{p["id"]}"'
        f' style="left:{p["x"]}px;top:{p["y"]}px"{marque_source}>{contenu}</div>'
    )


cartes_html = "\n".join(carte_page_html(p) for p in pages_brutes)

pages_pour_js = [{
    "id": p["id"], "ville": p["ville"], "annee": p["annee"], "titre": p["titre"],
    "publisher": p["publisher"], "graveur": p["graveur"], "famille": p["famille"],
    "url": p["url"], "source": p["source"],
} for p in pages_brutes]


def ligne_tableau(p):
    lien = f'<a href="{p["url"]}" target="_blank">voir</a>' if p["lien_html_ok"] else ""
    return (
        f'<tr><td>{p["annee"]}</td><td>{p["titre"]}</td><td>{p["ville"]}</td>'
        f'<td>{p["publisher"]}</td><td>{p["graveur"]}</td><td>{lien}</td></tr>'
    )


lignes_tableau = "\n".join(ligne_tableau(p) for p in sorted(pages_brutes, key=lambda p: p["annee"]))

print(len(cartes_html), "caracteres de cartes HTML generes")

642250 caracteres de cartes HTML generes


In [11]:
TEMPLATE_HTML = r"""<!DOCTYPE html>
<html lang="fr"><head><meta charset="utf-8">
<title>Vignette de plaques reemployees : Salomon / Solis</title>
<style>
  :root {
    --surface: #fffaf0; --texte-fort: #2b1e15; --texte-att: #6b5c4f; --trait: #d8cfc0;
    --contour-point: #3e2c23; --c-lien: #2a78d6; --c-salomon: #9a5b26; --c-solis: #2a6f6f;
  }
  @media (prefers-color-scheme: dark) {
    :root {
      --surface: #1a1a19; --texte-fort: #f2ece2; --texte-att: #c3baa9; --trait: #3a352c;
      --contour-point: #f2ece2; --c-lien: #3987e5; --c-salomon: #d99a56; --c-solis: #5cbcbc;
    }
  }
  body { margin:0; font-family:Georgia,serif; background:var(--surface); color:var(--texte-fort); }
  .page { max-width:1150px; margin:0 auto; padding:16px 20px 32px; }
  h1 { font-size:19px; margin:0 0 4px; }
  p.souschapo { font-size:13px; color:var(--texte-att); margin:0 0 6px; max-width:820px; }
  p.souschapo a { color:var(--c-lien); }

  .commandes { display:flex; align-items:center; gap:8px; font-size:12px; color:var(--texte-att);
    margin:10px 0 4px; }
  .bascule-famille { font-family:Georgia,serif; font-size:12px; background:none;
    border:1px solid var(--trait); color:var(--texte-fort); border-radius:4px; padding:5px 10px;
    cursor:pointer; }
  .bascule-famille[data-famille="salomon"].actif { background:var(--c-salomon); color:#fff; border-color:var(--c-salomon); }
  .bascule-famille[data-famille="solis"].actif { background:var(--c-solis); color:#fff; border-color:var(--c-solis); }

  .scene { position:relative; width:1100px; max-width:100%; height:900px; margin:6px auto 0; }
  .centre { position:absolute; left:50%; top:calc(50% - 10px); transform:translate(-50%,-50%);
    display:flex; flex-direction:column; align-items:center; gap:8px; width:380px; z-index:5; }
  .centre img { max-width:380px; max-height:380px; width:auto; height:auto;
    border:1px solid var(--contour-point); border-radius:4px; box-shadow:0 3px 14px rgba(0,0,0,.35);
    background:#efe6d3; }
  .legende-centre { font-size:11.5px; color:var(--texte-att); text-align:center; max-width:340px; }
  .legende-centre a { color:var(--c-lien); }
  .legende-centre b { color:var(--texte-fort); }

  .carte-page { position:absolute; width:150px; transform:translate(-50%,-50%); cursor:pointer;
    text-align:center; transition:transform .15s, filter .15s; }
  .carte-page img { width:150px; height:110px; object-fit:cover; object-position:top;
    border:2px solid var(--trait); border-radius:3px; box-shadow:0 1px 5px rgba(0,0,0,.3);
    display:block; }
  .carte-page.famille-salomon img { border-color:var(--c-salomon); }
  .carte-page.famille-solis img { border-color:var(--c-solis); }
  .carte-page:hover, .carte-page.famille-active { transform:translate(-50%,-50%) scale(1.08); z-index:4; }
  .carte-page.famille-active img { box-shadow:0 0 0 3px var(--surface), 0 0 0 5px currentColor, 0 3px 10px rgba(0,0,0,.4); }
  .carte-page.famille-salomon.famille-active img { color:var(--c-salomon); }
  .carte-page.famille-solis.famille-active img { color:var(--c-solis); }
  .carte-page[data-source="1"]::after { content:"vignette source"; position:absolute; top:-16px;
    left:50%; transform:translateX(-50%); font-size:9px; color:var(--texte-att); white-space:nowrap; }
  .pas-image { width:150px; height:110px; border:2px dashed var(--trait); border-radius:3px;
    display:flex; align-items:center; justify-content:center; font-size:10px; color:var(--texte-att);
    padding:4px; box-sizing:border-box; }

  button.bascule { font-family:Georgia,serif; font-size:12px; background:none;
    border:1px solid var(--trait); color:var(--texte-fort); border-radius:4px; padding:5px 10px;
    cursor:pointer; margin-top:10px; }
  table.tableau-detaille { width:100%; border-collapse:collapse; font-size:12px; margin:8px 0 6px;
    display:none; }
  table.tableau-detaille.visible { display:table; }
  table.tableau-detaille th, table.tableau-detaille td { text-align:left; padding:4px 8px;
    border-bottom:1px solid var(--trait); }
  table.tableau-detaille a { color:var(--c-lien); }

  .infobulle { position:absolute; pointer-events:none; background:var(--surface);
    border:1px solid var(--contour-point); border-radius:5px; padding:6px 10px; font-size:12px;
    max-width:260px; opacity:0; transition:opacity .1s; box-shadow:0 2px 8px rgba(0,0,0,.3); z-index:2000; }
  .infobulle.epinglee { pointer-events:auto; }
  .infobulle a { color:var(--c-lien); }
  .infobulle .fermer-infobulle { position:absolute; top:2px; right:6px; cursor:pointer;
    color:var(--texte-att); font-size:13px; }
</style></head><body>
<div class="page">
  <h1>Vignette de plaques reemployees : Salomon / Solis</h1>
  <p class="souschapo">Memes plaques gravees, orientation inversee : Bernard Salomon (Lyon,
    1557, <a href="https://gallica.bnf.fr/ark:/12148/btv1b2200047r/f10.item" target="_blank">page source</a>)
    et Virgil Solis (s.l., 1563, <a href="https://gallica.bnf.fr/ark:/12148/btv1b8454722p/f28.item" target="_blank">page source</a>).
    Cliquer une page recentre la vignette de sa famille de plaques ; le survol seul affiche un
    apercu sans lien.</p>
  <div class="commandes">
    <span>Vignette centrale :</span>
    <button class="bascule-famille" data-famille="salomon">Salomon (1557)</button>
    <button class="bascule-famille" data-famille="solis">Solis (1563)</button>
  </div>
  <div class="scene">
    <div class="centre">
      <img id="vignette-centrale" src="" alt="vignette centrale">
      <div class="legende-centre" id="legende-centre"></div>
    </div>
    __CARTES__
  </div>
  <button class="bascule" id="boutonTableau">Afficher le tableau detaille</button>
  <table class="tableau-detaille" id="tableauDetaille">
    <thead><tr><th>Annee</th><th>Titre</th><th>Ville</th><th>Editeur</th><th>Graveur</th><th>Lien</th></tr></thead>
    <tbody>
      __LIGNES_TABLEAU__
    </tbody>
  </table>
  <div class="infobulle" id="infobulle"></div>
</div>
<script>
  const pages = __PAGES__;
  const parId = {};
  pages.forEach(p => parId[p.id] = p);

  const VIGNETTE = { salomon: "__VIGNETTE_SALOMON__", solis: "__VIGNETTE_SOLIS__" };
  const LEGENDE = {
    salomon: 'Plaques de <b>Bernard Salomon</b> &mdash; Lyon, 1557. <a href="https://gallica.bnf.fr/ark:/12148/btv1b2200047r/f10.item" target="_blank">page source</a>',
    solis: 'Plaques de <b>Virgil Solis</b> &mdash; s.l., 1563 (memes plaques que Salomon, orientation inversee). <a href="https://gallica.bnf.fr/ark:/12148/btv1b8454722p/f28.item" target="_blank">page source</a>'
  };

  function definirCentre(famille) {
    document.getElementById('vignette-centrale').src = VIGNETTE[famille];
    document.getElementById('legende-centre').innerHTML = LEGENDE[famille];
    document.querySelectorAll('.carte-page').forEach(el => {
      el.classList.toggle('famille-active', el.classList.contains('famille-' + famille));
    });
    document.querySelectorAll('.bascule-famille').forEach(b => {
      b.classList.toggle('actif', b.dataset.famille === famille);
    });
  }

  function contenuApercu(p) {
    return '<b>' + p.titre + '</b><br><span>' + p.ville + ', ' + p.annee + '</span><br>' + p.graveur;
  }
  function contenuDetaille(p) {
    return contenuApercu(p) + '<br><i>' + p.publisher + '</i>' +
      (p.url ? '<br><a href="' + p.url + '" target="_blank">voir la page</a>' : '') +
      '<span class="fermer-infobulle" onclick="fermerInfobulle()">&times;</span>';
  }

  const infobulle = document.getElementById('infobulle');
  function positionnerInfobulle(ev) {
    infobulle.style.left = (ev.pageX + 14) + 'px';
    infobulle.style.top = (ev.pageY + 10) + 'px';
  }
  function fermerInfobulle() {
    infobulle.classList.remove('epinglee');
    infobulle.style.opacity = 0;
  }

  document.querySelectorAll('.carte-page').forEach(el => {
    const p = parId[el.dataset.i];
    el.addEventListener('mouseenter', (ev) => {
      if (infobulle.classList.contains('epinglee')) return;
      infobulle.innerHTML = contenuApercu(p);
      infobulle.style.opacity = 1;
      positionnerInfobulle(ev);
    });
    el.addEventListener('mousemove', (ev) => {
      if (infobulle.classList.contains('epinglee')) return;
      positionnerInfobulle(ev);
    });
    el.addEventListener('mouseleave', () => {
      if (infobulle.classList.contains('epinglee')) return;
      infobulle.style.opacity = 0;
    });
    el.addEventListener('click', (ev) => {
      ev.stopPropagation();
      definirCentre(p.famille);
      infobulle.innerHTML = contenuDetaille(p);
      infobulle.classList.add('epinglee');
      infobulle.style.opacity = 1;
      positionnerInfobulle(ev);
    });
  });

  document.querySelectorAll('.bascule-famille').forEach(b => {
    b.addEventListener('click', () => definirCentre(b.dataset.famille));
  });

  document.addEventListener('click', (ev) => {
    if (infobulle.classList.contains('epinglee') && !infobulle.contains(ev.target) && !ev.target.closest('.carte-page')) {
      fermerInfobulle();
    }
  });

  document.getElementById('boutonTableau').addEventListener('click', () => {
    document.getElementById('tableauDetaille').classList.toggle('visible');
  });

  definirCentre('salomon');
</script>
</body></html>"""

html_final = (TEMPLATE_HTML
    .replace("__CARTES__", cartes_html)
    .replace("__LIGNES_TABLEAU__", lignes_tableau)
    .replace("__PAGES__", json.dumps(pages_pour_js, ensure_ascii=False))
    .replace("__VIGNETTE_SALOMON__", vignette_salomon_b64)
    .replace("__VIGNETTE_SOLIS__", vignette_solis_b64))

CHEMIN_SORTIE = "../../resultats/Datavis/vignette_plaques.html"
os.makedirs(os.path.dirname(CHEMIN_SORTIE), exist_ok=True)
with open(CHEMIN_SORTIE, "w", encoding="utf-8") as f:
    f.write(html_final)

print(f"Ecrit : {CHEMIN_SORTIE} ({len(html_final)//1024} Ko)")

Ecrit : ../../resultats/Datavis/vignette_plaques.html (1169 Ko)
